In [1]:
import cv2

# Load model Haar Cascade untuk deteksi wajah
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Baca gambar
image = cv2.imread('smile.jpg')
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Deteksi wajah dalam gambar
faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

# Gambar kotak di sekitar wajah
for (x, y, w, h) in faces:
    cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)

cv2.namedWindow("image")
cv2.imshow("image", image)
cv2.waitKey(0)
cv2.destroyWindow('image')
cv2.waitKey(1)

2025-03-10 15:49:16.908 Python[8775:93827] +[IMKClient subclass]: chose IMKClient_Modern
2025-03-10 15:49:16.908 Python[8775:93827] +[IMKInputSession subclass]: chose IMKInputSession_Modern


-1

In [1]:
import cv2

# Inisialisasi kamera
cap = cv2.VideoCapture(0)

# Load model deteksi wajah
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

while True:
    ret, frame = cap.read()
    if not ret:
        print("⚠ Gagal menangkap frame!")
        break

    # Konversi ke grayscale untuk deteksi wajah
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5)

    # Gambar kotak hijau pada wajah yang terdeteksi
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Tampilkan video dengan deteksi wajah
    cv2.imshow('Face Detection', frame)

    # Tekan 'q' untuk keluar dari loop
    if cv2.waitKey(1) & 0xFF == ord('q'):
        cv2.destroyAllWindows()
        cv2.waitKey(1)
        break

# Lepaskan kamera dan tutup semua jendela
cap.release()

2025-03-10 16:18:55.635 Python[10155:119243] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.
2025-03-10 16:18:57.628 Python[10155:119243] +[IMKClient subclass]: chose IMKClient_Modern
2025-03-10 16:18:57.628 Python[10155:119243] +[IMKInputSession subclass]: chose IMKInputSession_Modern


: 

In [1]:
import cv2
import numpy as np

# Load detektor wajah dengan Haar Cascade
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def get_face_histogram(image):
    """Mendeteksi wajah dan mengembalikan histogram wajah sebagai fitur."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    if len(faces) == 0:
        return None  # Tidak ada wajah terdeteksi

    x, y, w, h = faces[0]  # Ambil wajah pertama yang terdeteksi
    face_roi = gray[y:y+h, x:x+w]  # Ambil Region of Interest (ROI) wajah

    # Hitung histogram wajah
    hist = cv2.calcHist([face_roi], [0], None, [256], [0, 256])
    hist = cv2.normalize(hist, hist).flatten()  # Normalisasi histogram
    return hist

# Load gambar referensi dan gambar target
image_ref = cv2.imread("smile.jpg")
image_target = cv2.imread("smile2.jpg")

# Ekstraksi fitur histogram wajah
hist_ref = get_face_histogram(image_ref)
hist_target = get_face_histogram(image_target)

# Periksa apakah wajah berhasil dideteksi di kedua gambar
if hist_ref is None or hist_target is None:
    print("⚠ Wajah tidak terdeteksi di salah satu gambar!")
else:
    # Bandingkan wajah dengan metode korelasi histogram
    similarity = cv2.compareHist(hist_ref, hist_target, cv2.HISTCMP_CORREL)

    # Threshold untuk menentukan kemiripan wajah (1 = mirip, 0 = tidak mirip)
    threshold = 0.7  
    if similarity > threshold:
        print("✅ Wajah cocok!")
    else:
        print("❌ Wajah tidak cocok!")

    print(f"🔍 Skor kesamaan wajah: {similarity:.4f}")


✅ Wajah cocok!
🔍 Skor kesamaan wajah: 0.8588


In [3]:
import cv2
import dlib
import numpy as np

# Load model deteksi wajah Haar Cascade
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

# Load model Dlib untuk encoding wajah
sp = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
face_rec_model = dlib.face_recognition_model_v1("dlib_face_recognition_resnet_model_v1.dat")

def get_face_encoding(image, face):
    """Mengembalikan encoding wajah yang terdeteksi."""
    x, y, w, h = face
    face_rect = dlib.rectangle(x, y, x + w, y + h)
    shape = sp(image, face_rect)
    encoding = np.array(face_rec_model.compute_face_descriptor(image, shape))
    return encoding

# Load gambar wajah yang dikenal
image_known = cv2.imread("smile.jpg")
image_known_rgb = cv2.cvtColor(image_known, cv2.COLOR_BGR2RGB)

# Ambil encoding wajah yang dikenal
gray_known = cv2.cvtColor(image_known, cv2.COLOR_BGR2GRAY)
faces_known = face_cascade.detectMultiScale(gray_known, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

if len(faces_known) > 0:
    known_encoding = get_face_encoding(image_known_rgb, faces_known[0])
    known_encodings = [known_encoding]
    known_names = ["Nama Anda"]
else:
    print("⚠ Tidak ada wajah yang terdeteksi dalam gambar yang dikenal!")
    exit()

# Inisialisasi webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        print("⚠ Gagal menangkap frame!")
        break

    # Konversi ke grayscale untuk deteksi Haar Cascade
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    for face in faces:
        x, y, w, h = face

        # Ambil encoding wajah
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        face_encoding = get_face_encoding(rgb_frame, face)

        # Bandingkan dengan wajah yang dikenal
        name = "Unknown"
        face_distances = [np.linalg.norm(face_encoding - known_enc) for known_enc in known_encodings]
        best_match_index = np.argmin(face_distances)

        if face_distances[best_match_index] < 0.6:
            name = known_names[best_match_index]

        # Gambar kotak di sekitar wajah
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # Tampilkan nama di bawah kotak
        cv2.putText(frame, name, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # Tampilkan frame
    cv2.imshow("Face Recognition (Haar Cascade + Dlib)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        cv2.destroyAllWindows()
        cv2.waitKey(1)
        break

# Lepaskan kamera dan tutup jendela
cap.release()


2025-03-10 16:58:10.338 Python[11349:142982] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.
2025-03-10 16:59:39.459 Python[11349:142982] +[IMKClient subclass]: chose IMKClient_Modern
2025-03-10 16:59:39.459 Python[11349:142982] +[IMKInputSession subclass]: chose IMKInputSession_Modern
